# Pontuação Automática por Aspectos em Avaliações de Praias

Este notebook implementa o módulo de Análise de Sentimento Baseada em Aspectos (ABSA) para avaliações de praias coletadas no TripAdvisor.

O objetivo é transformar comentários livres em uma base estruturada, na qual cada praia passa a ter notas específicas por aspecto, acompanhadas de evidências textuais extraídas das próprias avaliações.

A pipeline implementada:

1. carregamento das avaliações do TripAdvisor;
2. concatenação de `title + text` em uma única avaliação;
3. detecção dos aspectos mencionados com apoio do Gemini;
4. atribuição de nota de 1 a 5 para cada aspecto citado;
5. extração de evidências textuais que justificam as notas;
6. agregação dos resultados por praia e aspecto;
7. exportação das bases finais em CSV.

## Executado no Colab


In [1]:
!pip install -q -U google-genai

from google.colab import drive, userdata
from google import genai
from pathlib import Path
import json
import os
import re
import time

import numpy as np
import pandas as pd

drive.mount('/content/drive')

^C


ModuleNotFoundError: No module named 'google.colab'

## 1. Configuração do ambiente e dos dados

Esta etapa prepara o ambiente de execução no Google Colab, monta o Google Drive, localiza o arquivo final de avaliações e cria o cliente da API Gemini.

A base utilizada é `avaliacoes_praias_final.csv`, que já contém as avaliações consolidadas e pré-processadas no pipeline textual do projeto.

In [8]:
BASE_DIR = Path('/content/drive/MyDrive/NLP---Natural-Language-Processing')
CAMINHO_CSV = BASE_DIR / 'dados' / 'finais' / 'avaliacoes_praias_final.csv'
PASTA_SAIDA = BASE_DIR / 'dados' / 'absa'

assert BASE_DIR.exists(), f'Pasta nao encontrada: {BASE_DIR}'
assert CAMINHO_CSV.exists(), f'CSV nao encontrado: {CAMINHO_CSV}'

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
assert GEMINI_API_KEY, 'Crie um Secret no Colab chamado GEMINI_API_KEY.'

client = genai.Client(api_key=GEMINI_API_KEY)

print('BASE_DIR:', BASE_DIR)
print('CSV:', CAMINHO_CSV)
print('PASTA_SAIDA:', PASTA_SAIDA)

BASE_DIR: /content/drive/MyDrive/NLP---Natural-Language-Processing
CSV: /content/drive/MyDrive/NLP---Natural-Language-Processing/dados/finais/avaliacoes_praias_final.csv
PASTA_SAIDA: /content/drive/MyDrive/NLP---Natural-Language-Processing/dados/absa


## 2. Definição dos aspectos

Os aspectos abaixo representam as dimensões avaliadas pelo módulo ABSA. Eles foram definidos de acordo com o escopo do trabalho e representam características relevantes para comparar praias a partir das avaliações dos turistas.

In [9]:
ASPECTOS = {
    'seguranca': {
        'nome': 'Seguranca',
        'descricao': 'seguranca, perigo, salva-vidas, risco, assalto, sensacao de protecao'
    },
    'limpeza': {
        'nome': 'Limpeza',
        'descricao': 'limpeza da praia, areia, agua, lixo, sujeira, esgoto, poluicao'
    },
    'infraestrutura': {
        'nome': 'Infraestrutura',
        'descricao': 'banheiro, chuveiro, calcadao, sinalizacao, estrutura fisica, quiosques'
    },
    'mar_banho': {
        'nome': 'Mar / banho',
        'descricao': 'condicoes do mar e banho, ondas, correnteza, agua, mar calmo ou agitado'
    },
    'familia_criancas': {
        'nome': 'Familia / criancas',
        'descricao': 'adequacao para familias, criancas, idosos, filhos e passeio familiar'
    },
    'tranquilidade': {
        'nome': 'Tranquilidade',
        'descricao': 'calma, sossego, lotacao, barulho, praia cheia ou reservada'
    },
    'acesso_estacionamento': {
        'nome': 'Acesso / estacionamento',
        'descricao': 'acesso, trilha, estrada, escada, chegada, vagas e estacionamento'
    },
    'beleza_natural': {
        'nome': 'Beleza natural',
        'descricao': 'beleza, paisagem, vista, natureza, praia bonita, visual, paraiso'
    },
    'comercio_servicos': {
        'nome': 'Comercio / servicos',
        'descricao': 'bares, restaurantes, quiosques, ambulantes, comida, bebida e servicos'
    },
    'custo_beneficio': {
        'nome': 'Custo-beneficio',
        'descricao': 'preco, valor, caro, barato, taxas e custo-beneficio'
    },
}

ASPECTOS

{'seguranca': {'nome': 'Seguranca',
  'descricao': 'seguranca, perigo, salva-vidas, risco, assalto, sensacao de protecao'},
 'limpeza': {'nome': 'Limpeza',
  'descricao': 'limpeza da praia, areia, agua, lixo, sujeira, esgoto, poluicao'},
 'infraestrutura': {'nome': 'Infraestrutura',
  'descricao': 'banheiro, chuveiro, calcadao, sinalizacao, estrutura fisica, quiosques'},
 'mar_banho': {'nome': 'Mar / banho',
  'descricao': 'condicoes do mar e banho, ondas, correnteza, agua, mar calmo ou agitado'},
 'familia_criancas': {'nome': 'Familia / criancas',
  'descricao': 'adequacao para familias, criancas, idosos, filhos e passeio familiar'},
 'tranquilidade': {'nome': 'Tranquilidade',
  'descricao': 'calma, sossego, lotacao, barulho, praia cheia ou reservada'},
 'acesso_estacionamento': {'nome': 'Acesso / estacionamento',
  'descricao': 'acesso, trilha, estrada, escada, chegada, vagas e estacionamento'},
 'beleza_natural': {'nome': 'Beleza natural',
  'descricao': 'beleza, paisagem, vista, na

## 3. Funções do pipeline

As funções a seguir implementam a pipeline completa em um único notebook: leitura da base, construção da avaliação concatenada, montagem do prompt, chamada ao Gemini, tratamento da resposta em JSON, agregação dos resultados e exportação dos arquivos finais.

In [10]:
def carregar_base(caminho_csv):
    return pd.read_csv(caminho_csv, encoding='utf-8-sig')


def texto_limpo(valor):
    if pd.isna(valor):
        return ''
    return re.sub(r'\s+', ' ', str(valor)).strip()


def criar_review(row):
    titulo = texto_limpo(row.get('title', ''))
    texto = texto_limpo(row.get('text', ''))
    if titulo and texto:
        return f'{titulo}. {texto}'
    return titulo or texto


def montar_prompt_gemini(itens):
    return f'''
Voce e um sistema de ABSA (Analise de Sentimento Baseada em Aspectos) para avaliacoes de praias em portugues do Brasil.

Para cada review, faca exatamente isto:
1. Identifique quais aspectos sao mencionados explicitamente ou claramente sugeridos.
2. Para cada aspecto mencionado, extraia uma evidencia curta do texto.
3. Para cada aspecto mencionado, atribua uma nota de 1 a 5.
4. Nao retorne aspectos que nao aparecem na review.
5. Nao invente evidencia.

Escala da nota:
1 = muito negativo
2 = negativo
3 = neutro, misto ou insuficiente
4 = positivo
5 = muito positivo

Use a nota geral da review apenas como contexto fraco. A nota do aspecto deve vir principalmente da evidencia textual.

Aspectos permitidos, com codigos fixos:
{json.dumps(ASPECTOS, ensure_ascii=False, indent=2)}

Responda APENAS um JSON valido, sem markdown, no formato:
[
  {{
    "review_id": 1,
    "aspecto": "limpeza",
    "nota_aspecto": 2,
    "sentimento": "negativo",
    "confianca": 0.86,
    "evidencia": "agua com aspecto sujo",
    "justificativa": "A evidencia descreve sujeira na agua."
  }}
]

Valores permitidos para sentimento: muito_negativo, negativo, neutro, positivo, muito_positivo.
O campo aspecto deve ser exatamente um dos codigos permitidos.
A confianca deve ser numero entre 0 e 1.

Reviews para analisar:
{json.dumps(itens, ensure_ascii=False, indent=2)}
'''.strip()


def extrair_json_resposta(texto):
    texto = (texto or '').strip()
    if texto.startswith('```'):
        texto = re.sub(r'^```(?:json)?', '', texto, flags=re.IGNORECASE).strip()
        texto = re.sub(r'```$', '', texto).strip()

    try:
        saida = json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'\[[\s\S]*\]', texto)
        if not match:
            raise
        saida = json.loads(match.group(0))

    if isinstance(saida, dict):
        saida = [saida]
    if not isinstance(saida, list):
        raise ValueError('A resposta do Gemini nao retornou uma lista JSON.')
    return saida


def analisar_lote_gemini(client, itens, modelo='gemini-3.1-flash-lite', tentativas=2, pausa_erro=10):
    prompt = montar_prompt_gemini(itens)
    ultimo_erro = None

    for tentativa in range(1, tentativas + 1):
        try:
            resposta = client.models.generate_content(
                model=modelo,
                contents=prompt,
            )
            return extrair_json_resposta(resposta.text)
        except Exception as erro:
            ultimo_erro = erro
            if tentativa < tentativas:
                print(f'Tentativa {tentativa} falhou. Aguardando {pausa_erro}s antes de tentar novamente...')
                time.sleep(pausa_erro)

    raise RuntimeError(f'Falha ao analisar lote com Gemini: {ultimo_erro}')


def salvar_checkpoint(linhas, reviews_processadas, caminho_parcial, caminho_progresso):
    caminho_parcial = Path(caminho_parcial)
    caminho_progresso = Path(caminho_progresso)
    caminho_parcial.parent.mkdir(parents=True, exist_ok=True)

    df_parcial = pd.DataFrame(linhas)
    if not df_parcial.empty:
        df_parcial = df_parcial.drop_duplicates(subset=['review_id', 'aspecto', 'evidencia'])
    df_parcial.to_csv(caminho_parcial, index=False, encoding='utf-8-sig')

    df_progresso = pd.DataFrame({'review_id': sorted(set(int(x) for x in reviews_processadas))})
    df_progresso.to_csv(caminho_progresso, index=False, encoding='utf-8-sig')


def carregar_checkpoint(caminho_parcial, caminho_progresso):
    caminho_parcial = Path(caminho_parcial)
    caminho_progresso = Path(caminho_progresso)

    linhas = []
    reviews_processadas = set()

    if caminho_parcial.exists():
        df_parcial = pd.read_csv(caminho_parcial, encoding='utf-8-sig')
        linhas = df_parcial.to_dict('records')
        print(f'Checkpoint de resultados carregado: {len(df_parcial)} linhas review x aspecto.')

    if caminho_progresso.exists():
        df_progresso = pd.read_csv(caminho_progresso, encoding='utf-8-sig')
        reviews_processadas = set(df_progresso['review_id'].dropna().astype(int).tolist())
        print(f'Checkpoint de progresso carregado: {len(reviews_processadas)} reviews ja processadas.')
    elif linhas:
        reviews_processadas = set(pd.DataFrame(linhas)['review_id'].dropna().astype(int).tolist())
        print('Arquivo de progresso nao encontrado; progresso reconstruido a partir do CSV parcial.')

    return linhas, reviews_processadas


def gerar_tabela_aspectos_gemini(
    df,
    client,
    modelo='gemini-3.1-flash-lite',
    limite=None,
    tamanho_lote=20,
    pausa_lote=10,
    caminho_parcial=None,
    caminho_progresso=None,
):
    base = df.reset_index(drop=True).copy()
    if limite is not None:
        base = base.head(limite)

    if caminho_parcial is None:
        caminho_parcial = PASTA_SAIDA / 'avaliacoes_aspectos_gemini_parcial.csv'
    if caminho_progresso is None:
        caminho_progresso = PASTA_SAIDA / 'reviews_processadas_gemini.csv'

    linhas, reviews_processadas = carregar_checkpoint(caminho_parcial, caminho_progresso)
    total = len(base)

    for inicio in range(0, total, tamanho_lote):
        lote_original = base.iloc[inicio:inicio + tamanho_lote]
        lote = lote_original[[int(idx + 1) not in reviews_processadas for idx in lote_original.index]]

        if lote.empty:
            print(f'Lote {inicio + 1}-{min(inicio + tamanho_lote, total)} ja estava processado. Pulando...')
            continue

        itens = []
        ids_lote = []

        for idx, row in lote.iterrows():
            review_id = int(idx + 1)
            ids_lote.append(review_id)
            itens.append({
                'review_id': review_id,
                'praia': row.get('praia'),
                'rating_geral': int(row.get('rating')),
                'review': criar_review(row),
            })

        try:
            resultados = analisar_lote_gemini(client, itens, modelo=modelo)
        except Exception:
            salvar_checkpoint(linhas, reviews_processadas, caminho_parcial, caminho_progresso)
            raise

        por_review = {int(row.name + 1): row for _, row in lote.iterrows()}

        for item in resultados:
            try:
                review_id = int(item.get('review_id'))
            except (TypeError, ValueError):
                continue

            row = por_review.get(review_id)
            if row is None:
                continue

            aspecto = str(item.get('aspecto', '')).strip()
            if aspecto not in ASPECTOS:
                continue

            try:
                nota = int(item.get('nota_aspecto'))
            except (TypeError, ValueError):
                nota = 3
            nota = max(1, min(5, nota))

            try:
                confianca = float(item.get('confianca', np.nan))
            except (TypeError, ValueError):
                confianca = np.nan
            if np.isfinite(confianca):
                confianca = max(0.0, min(1.0, confianca))

            linhas.append({
                'review_id': review_id,
                'praia': row.get('praia'),
                'rating_geral': row.get('rating'),
                'title': row.get('title'),
                'date': row.get('date'),
                'review': criar_review(row),
                'aspecto': aspecto,
                'aspecto_nome': ASPECTOS[aspecto]['nome'],
                'nota_aspecto': nota,
                'sentimento': str(item.get('sentimento', '')),
                'confianca': round(confianca, 3) if np.isfinite(confianca) else np.nan,
                'evidencia': str(item.get('evidencia', ''))[:500],
                'justificativa': str(item.get('justificativa', ''))[:500],
                'metodo': 'gemini_direto',
                'modelo_gemini': modelo,
            })

        reviews_processadas.update(ids_lote)
        salvar_checkpoint(linhas, reviews_processadas, caminho_parcial, caminho_progresso)

        print(f'Gemini: {len(reviews_processadas)}/{total} reviews processadas. Checkpoint salvo.')
        if pausa_lote:
            time.sleep(pausa_lote)

    df_resultado = pd.DataFrame(linhas)
    if not df_resultado.empty:
        df_resultado = df_resultado.drop_duplicates(subset=['review_id', 'aspecto', 'evidencia']).reset_index(drop=True)
    return df_resultado


def agregar_por_praia(df_aspectos, df_original):
    if df_aspectos.empty:
        return pd.DataFrame()

    grupo = df_aspectos.groupby(['praia', 'aspecto', 'aspecto_nome'], dropna=False)
    agg = grupo.agg(
        n_mencoes=('nota_aspecto', 'size'),
        nota_media=('nota_aspecto', 'mean'),
        nota_mediana=('nota_aspecto', 'median'),
        pct_negativas=('nota_aspecto', lambda s: (s <= 2).mean() * 100),
        pct_positivas=('nota_aspecto', lambda s: (s >= 4).mean() * 100),
        confianca_media=('confianca', 'mean'),
    ).reset_index()

    exemplos = (
        df_aspectos.sort_values(['praia', 'aspecto', 'confianca'], ascending=[True, True, False])
        .groupby(['praia', 'aspecto'])['evidencia']
        .apply(lambda s: ' | '.join(dict.fromkeys([x for x in s.astype(str) if x.strip()]))[:500])
        .reset_index(name='exemplos_evidencia')
    )
    agg = agg.merge(exemplos, on=['praia', 'aspecto'], how='left')

    totais = df_original.groupby('praia').size().rename('n_avaliacoes').reset_index()
    agg = agg.merge(totais, on='praia', how='left')
    agg['pct_mencoes'] = agg['n_mencoes'] / agg['n_avaliacoes'] * 100

    for coluna in ['nota_media', 'nota_mediana', 'pct_negativas', 'pct_positivas', 'confianca_media', 'pct_mencoes']:
        agg[coluna] = agg[coluna].round(2)

    return agg[[
        'praia', 'aspecto', 'aspecto_nome', 'n_avaliacoes', 'n_mencoes', 'pct_mencoes',
        'nota_media', 'nota_mediana', 'pct_negativas', 'pct_positivas', 'confianca_media', 'exemplos_evidencia'
    ]].sort_values(['praia', 'aspecto']).reset_index(drop=True)


def resumo_cobertura(df_aspectos, total_reviews):
    if df_aspectos.empty:
        return pd.DataFrame()

    resumo = (
        df_aspectos.groupby(['aspecto', 'aspecto_nome'])
        .agg(
            n_mencoes=('review_id', 'count'),
            n_reviews=('review_id', 'nunique'),
            nota_media=('nota_aspecto', 'mean'),
            pct_negativas=('nota_aspecto', lambda s: (s <= 2).mean() * 100),
            pct_positivas=('nota_aspecto', lambda s: (s >= 4).mean() * 100),
        )
        .reset_index()
    )
    resumo['pct_reviews'] = resumo['n_reviews'] / total_reviews * 100

    for coluna in ['nota_media', 'pct_negativas', 'pct_positivas', 'pct_reviews']:
        resumo[coluna] = resumo[coluna].round(2)

    return resumo.sort_values('n_reviews', ascending=False).reset_index(drop=True)


def rodar_pipeline_gemini(
    caminho_csv,
    client,
    modelo='gemini-3.1-flash-lite',
    limite=None,
    tamanho_lote=20,
    pausa_lote=10,
):
    df_base = carregar_base(caminho_csv)

    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    caminho_review_aspecto = PASTA_SAIDA / 'avaliacoes_aspectos_gemini.csv'
    caminho_praia_aspecto = PASTA_SAIDA / 'praia_aspecto_scores_gemini.csv'
    caminho_parcial = PASTA_SAIDA / 'avaliacoes_aspectos_gemini_parcial.csv'
    caminho_progresso = PASTA_SAIDA / 'reviews_processadas_gemini.csv'

    df_aspectos = gerar_tabela_aspectos_gemini(
        df_base,
        client=client,
        modelo=modelo,
        limite=limite,
        tamanho_lote=tamanho_lote,
        pausa_lote=pausa_lote,
        caminho_parcial=caminho_parcial,
        caminho_progresso=caminho_progresso,
    )
    df_praias = agregar_por_praia(df_aspectos, df_base)

    df_aspectos.to_csv(caminho_review_aspecto, index=False, encoding='utf-8-sig')
    df_praias.to_csv(caminho_praia_aspecto, index=False, encoding='utf-8-sig')

    print('Reviews na base:', len(df_base))
    print('Reviews esperadas nesta execucao:', len(df_base.head(limite)) if limite is not None else len(df_base))
    print('Linhas review x aspecto:', len(df_aspectos))
    print('Linhas praia x aspecto:', len(df_praias))
    print('Arquivo review x aspecto:', caminho_review_aspecto)
    print('Arquivo praia x aspecto:', caminho_praia_aspecto)
    print('Checkpoint parcial:', caminho_parcial)
    print('Checkpoint de progresso:', caminho_progresso)

    return df_base, df_aspectos, df_praias


## 4. Execução do pipeline

Nesta etapa, as avaliações são enviadas ao Gemini em pequenos lotes. Para cada avaliação, o modelo identifica os aspectos citados, atribui uma nota de 1 a 5 e retorna uma evidência textual que justifica a pontuação.

Nesta versão final, o parâmetro `LIMITE_REVIEWS` é definido como `None`, indicando que toda a base deve ser processada. Como cada lote é enviado ao Gemini, a execução completa pode consumir cota da API e levar mais tempo.

O notebook salva progresso após cada lote em dois arquivos de checkpoint: `avaliacoes_aspectos_gemini_parcial.csv`, com os resultados já recebidos, e `reviews_processadas_gemini.csv`, com os identificadores das avaliações já concluídas. Se a execução parar por limite de tempo, erro ou cota, a próxima execução continua a partir das avaliações ainda não processadas.


In [11]:
MODELO_GEMINI = 'gemini-3.1-flash-lite'
LIMITE_REVIEWS = None   # processa toda a base
TAMANHO_LOTE = 20       # menos requisicoes do que lotes pequenos
PAUSA_LOTE = 10         # intervalo entre lotes para reduzir risco de rate limit

df_base, df_aspectos_gemini, df_praias_gemini = rodar_pipeline_gemini(
    CAMINHO_CSV,
    client=client,
    modelo=MODELO_GEMINI,
    limite=LIMITE_REVIEWS,
    tamanho_lote=TAMANHO_LOTE,
    pausa_lote=PAUSA_LOTE,
)

display(df_aspectos_gemini.head(20))
display(df_praias_gemini.head(20))


Checkpoint de resultados carregado: 13230 linhas review x aspecto.
Checkpoint de progresso carregado: 6212 reviews ja processadas.
Lote 1-20 ja estava processado. Pulando...
Lote 21-40 ja estava processado. Pulando...
Lote 41-60 ja estava processado. Pulando...
Lote 61-80 ja estava processado. Pulando...
Lote 81-100 ja estava processado. Pulando...
Lote 101-120 ja estava processado. Pulando...
Lote 121-140 ja estava processado. Pulando...
Lote 141-160 ja estava processado. Pulando...
Lote 161-180 ja estava processado. Pulando...
Lote 181-200 ja estava processado. Pulando...
Lote 201-220 ja estava processado. Pulando...
Lote 221-240 ja estava processado. Pulando...
Lote 241-260 ja estava processado. Pulando...
Lote 261-280 ja estava processado. Pulando...
Lote 281-300 ja estava processado. Pulando...
Lote 301-320 ja estava processado. Pulando...
Lote 321-340 ja estava processado. Pulando...
Lote 341-360 ja estava processado. Pulando...
Lote 361-380 ja estava processado. Pulando...
Lote 

,review_id,praia,rating_geral,title,date,review,aspecto,aspecto_nome,nota_aspecto,sentimento,confianca,evidencia,justificativa,metodo,modelo_gemini
0,1,praia_alegre,5,Otima Cidade para passeios em familia.,22 de abril de 2025,Otima Cidade para passeios em familia.. Cidade...,familia_criancas,Familia / criancas,5,muito_positivo,0.95,Otima Cidade para passeios em familia,O autor destaca positivamente a adequação para...,gemini_direto,gemini-3.1-flash-lite
1,1,praia_alegre,5,Otima Cidade para passeios em familia.,22 de abril de 2025,Otima Cidade para passeios em familia.. Cidade...,tranquilidade,Tranquilidade,5,muito_positivo,0.90,Cidade tranquila,Menciona a tranquilidade como característica p...,gemini_direto,gemini-3.1-flash-lite
2,1,praia_alegre,5,Otima Cidade para passeios em familia.,22 de abril de 2025,Otima Cidade para passeios em familia.. Cidade...,comercio_servicos,Comercio / servicos,4,positivo,0.90,"boas praias, restaurantes",Avalia positivamente a oferta de restaurantes.,gemini_direto,gemini-3.1-flash-lite
3,1,praia_alegre,5,Otima Cidade para passeios em familia.,22 de abril de 2025,Otima Cidade para passeios em familia.. Cidade...,acesso_estacionamento,Acesso / estacionamento,3,neutro,0.85,Há algumas ruas de terra ainda,O aspecto é neutro pois menciona terra mas diz...,gemini_direto,gemini-3.1-flash-lite
4,2,praia_alegre,1,Praia abandonada,8 de janeiro de 2025,"Praia abandonada. Praia suja , mal cuidada , l...",limpeza,Limpeza,1,muito_negativo,1.00,"Praia suja , mal cuidada , lixo espalhado",Descrição clara de falta de limpeza.,gemini_direto,gemini-3.1-flash-lite
5,2,praia_alegre,1,Praia abandonada,8 de janeiro de 2025,"Praia abandonada. Praia suja , mal cuidada , l...",mar_banho,Mar / banho,1,muito_negativo,1.00,"Água marrom , impropria para banho",Relata péssimas condições da água.,gemini_direto,gemini-3.1-flash-lite
6,3,praia_alegre,5,Caro e fica em praia impropria para banho,4 de outubro de 2024,Caro e fica em praia impropria para banho. Lug...,custo_beneficio,Custo-beneficio,1,muito_negativo,0.95,preço de consumo abusivo,Afirma que o preço é abusivo e não compensa.,gemini_direto,gemini-3.1-flash-lite
7,3,praia_alegre,5,Caro e fica em praia impropria para banho,4 de outubro de 2024,Caro e fica em praia impropria para banho. Lug...,mar_banho,Mar / banho,1,muito_negativo,1.00,praia impropria para banho,Menciona que a praia não é adequada para banho.,gemini_direto,gemini-3.1-flash-lite
8,4,praia_alegre,5,Tardena Praia Alegre,13 de junho de 2024,Tardena Praia Alegre. Praia muito legal para i...,familia_criancas,Familia / criancas,5,muito_positivo,0.95,Praia muito legal para ir com a família,Afirma que a praia é ideal para família.,gemini_direto,gemini-3.1-flash-lite
9,4,praia_alegre,5,Tardena Praia Alegre,13 de junho de 2024,Tardena Praia Alegre. Praia muito legal para i...,mar_banho,Mar / banho,5,muito_positivo,0.95,mar calmo para as crianças,Avalia positivamente as condições do mar para ...,gemini_direto,gemini-3.1-flash-lite


,praia,aspecto,aspecto_nome,n_avaliacoes,n_mencoes,pct_mencoes,nota_media,nota_mediana,pct_negativas,pct_positivas,confianca_media,exemplos_evidencia
0,praia_alegre,acesso_estacionamento,Acesso / estacionamento,112,12,10.71,3.00,3.0,33.33,33.33,0.94,sem estacionamento | não existe mais estaciona...
1,praia_alegre,beleza_natural,Beleza natural,112,21,18.75,4.76,5.0,0.00,95.24,0.95,Praia bonita | Praia linda | praia já era lind...
2,praia_alegre,comercio_servicos,Comercio / servicos,112,23,20.54,3.74,4.0,8.70,78.26,0.93,existem quiosques e restaurantes perto | Pouco...
3,praia_alegre,custo_beneficio,Custo-beneficio,112,3,2.68,3.33,4.0,33.33,66.67,0.93,preço de consumo abusivo | Todos os ambulantes...
4,praia_alegre,familia_criancas,Familia / criancas,112,34,30.36,4.91,5.0,0.00,100.00,0.96,Muito boa para aproveitar com as crianças | as...
5,praia_alegre,infraestrutura,Infraestrutura,112,30,26.79,4.20,4.5,13.33,86.67,0.95,O calçadão ficou lindo e atrativo | Sem nenhum...
6,praia_alegre,limpeza,Limpeza,112,41,36.61,2.90,3.0,48.78,43.90,0.96,"Praia suja , mal cuidada , lixo espalhado | qu..."
7,praia_alegre,mar_banho,Mar / banho,112,64,57.14,3.44,4.0,32.81,53.12,0.95,"Água marrom , impropria para banho | praia imp..."
8,praia_alegre,seguranca,Seguranca,112,5,4.46,2.80,2.0,60.00,40.00,0.97,revitalização trouxe para a praia a segurança ...
9,praia_alegre,tranquilidade,Tranquilidade,112,40,35.71,4.42,5.0,7.50,87.50,0.94,sossego | não ouvi aquela barulheira de som | ...


## 5. Resumo de cobertura

O resumo de cobertura mostra quantas vezes cada aspecto foi identificado pelo Gemini, qual foi a nota média atribuída e a proporção de avaliações positivas ou negativas para cada dimensão analisada.

In [12]:
total_processado = LIMITE_REVIEWS if LIMITE_REVIEWS is not None else len(df_base)
display(resumo_cobertura(df_aspectos_gemini, total_reviews=total_processado))

print('CSVs gerados em:')
print(PASTA_SAIDA / 'avaliacoes_aspectos_gemini.csv')
print(PASTA_SAIDA / 'praia_aspecto_scores_gemini.csv')

,aspecto,aspecto_nome,n_mencoes,n_reviews,nota_media,pct_negativas,pct_positivas,pct_reviews
0,mar_banho,Mar / banho,2335,2335,3.87,21.80,63.21,37.59
1,beleza_natural,Beleza natural,1924,1924,4.85,1.40,97.30,30.97
2,tranquilidade,Tranquilidade,1721,1718,4.37,10.98,82.28,27.66
3,comercio_servicos,Comercio / servicos,1646,1644,3.96,14.88,75.52,26.46
4,limpeza,Limpeza,1599,1599,4.32,16.32,80.80,25.74
5,infraestrutura,Infraestrutura,1244,1243,3.51,32.56,60.93,20.01
6,familia_criancas,Familia / criancas,1154,1154,4.50,12.13,87.26,18.58
7,acesso_estacionamento,Acesso / estacionamento,913,912,2.98,49.84,36.25,14.68
8,seguranca,Seguranca,428,427,3.94,23.36,70.33,6.87
9,custo_beneficio,Custo-beneficio,266,266,3.73,27.82,63.16,4.28


CSVs gerados em:
/content/drive/MyDrive/NLP---Natural-Language-Processing/dados/absa/avaliacoes_aspectos_gemini.csv
/content/drive/MyDrive/NLP---Natural-Language-Processing/dados/absa/praia_aspecto_scores_gemini.csv
